# 02 · Sync Measurements to Lakebase (Postgres)

**Lakebase project:** `projects/oktex/branches/production/endpoints/primary` (Autoscaling tier)  
**Database:** `oktex`

The OkTex measurement tables are synced from the local generator (which mirrors the Delta tables built in notebook 01) into Lakebase Postgres. The Databricks App reads these tables **live** at request time. Verification queries below run over `psql` against the live endpoint using a short-lived OAuth credential.

### 1. Run the Delta → Lakebase sync and verify (live)

In [ ]:
import sync_lakebase
sync_lakebase.main()

# Lakebase sync -> ep-ancient-recipe-d2pmfxiq.database.us-east-1.cloud.databricks.com  db=oktex

## Step 1: create database
database 'oktex' ready

## Step 2: create tables and load rows
tables created and loaded

## Step 3: verification (live psql against Lakebase)
### row counts
table_name        | rows 
-------------------------+------
 dim_meters              |   20
 fact_daily_measurements | 1200
 pipeline_segments       |   14
(3 rows)

### date coverage
first_day  |  last_day  | days 
------------+------------+------
 2026-06-30 | 2026-08-28 |   60
(1 row)

### today's flow by meter type
meter_type  | meters | total_actual_dth 
--------------+--------+------------------
 DELIVERY     |      8 |           203506
 INTERCONNECT |      5 |           256926
 RECEIPT      |      7 |           449628
(3 rows)

### sample: today's top 8 meters
meter_id |      meter_name      |  meter_type  | actual_dth | pressure_psig | variance_pct 
----------+----------------------+--------------+----

### 2. Confirm the app's read view exists and returns today's rows

In [ ]:
host, token, email = sync_lakebase.conn_info()
r = sync_lakebase.psql(host, token, email, 'oktex',
    "SELECT meter_id, meter_name, meter_type, actual_dth, pressure_psig "
    "FROM v_latest_measurements ORDER BY actual_dth DESC LIMIT 5;")
print(r.stdout)

 meter_id |    meter_name     |  meter_type  | actual_dth | pressure_psig 
----------+-------------------+--------------+------------+---------------
 OKT-010  | Perryton Receipt  | RECEIPT      |      96305 |         806.5
 OKT-001  | Levelland Receipt | RECEIPT      |      72494 |         921.5
 OKT-013  | Woodward Hub      | INTERCONNECT |      72320 |         783.1
 OKT-008  | Carson Receipt    | RECEIPT      |      69732 |         863.2
 OKT-015  | Fairview Receipt  | RECEIPT      |      57992 |           708
(5 rows)


